In [ ]:
import gym
import numpy as np
import torch
import torch.nn as nn
from torch.distributions import Normal

# Policy Network (Actor)
class PolicyNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, action_std_init=0.6, device='cpu'):
        super(PolicyNetwork, self).__init__()
        self.device = torch.device(device)
        self.actor = nn.Sequential(
            nn.Linear(state_dim, 64), nn.LeakyReLU(),
            nn.Linear(64, 64), nn.LeakyReLU(),
            nn.Linear(64, action_dim)
        ).to(self.device)
        self.action_var = torch.full((action_dim,), action_std_init * action_std_init).to(self.device)

    def forward(self):
        raise NotImplementedError

    def act(self, state):
        state = state.to(self.device)
        action_mean = self.actor(state)
        dist = Normal(action_mean, torch.sqrt(self.action_var))
        action = dist.sample()
        action_logprob = dist.log_prob(action).sum(dim=-1)
        return action.detach(), action_logprob.detach()

    def evaluate(self, state, action):
        state = state.to(self.device)
        action = action.to(self.device)
        action_mean = self.actor(state)
        dist = Normal(action_mean, torch.sqrt(self.action_var))
        action_logprobs = dist.log_prob(action).sum(dim=-1)
        dist_entropy = dist.entropy().sum(dim=-1)
        return action_logprobs, dist_entropy

# Value Network (Critic)
class ValueNetwork(nn.Module):
    def __init__(self, state_dim, device='cpu'):
        super(ValueNetwork, self).__init__()
        self.device = torch.device(device)
        self.critic = nn.Sequential(
            nn.Linear(state_dim, 64), nn.LeakyReLU(),
            nn.Linear(64, 64), nn.LeakyReLU(),
            nn.Linear(64, 1)
        ).to(self.device)

    def forward(self, state):
        state = state.to(self.device)
        return self.critic(state).squeeze()

# PPO Agent with Clipped Surrogate and GAE
class PPO:
    def __init__(self, state_dim, action_dim,
                 lr_actor=3e-4, lr_critic=3e-4
                 gamma=0.99, lam=0.95,
                 K_epochs=80, eps_clip=0.2,
                 device='cpu'):
        self.gamma = gamma
        self.lam = lam
        self.eps_clip = eps_clip
        self.K_epochs = K_epochs
        self.device = torch.device(device)

        self.policy = PolicyNetwork(state_dim, action_dim, device=device)
        self.policy_old = PolicyNetwork(state_dim, action_dim, device=device)
        self.policy_old.load_state_dict(self.policy.state_dict())
        self.value_net = ValueNetwork(state_dim, device=device)

        self.policy_optimizer = torch.optim.Adam(self.policy.parameters(), lr=lr_actor, weight_decay=1e-4)
        self.value_optimizer = torch.optim.Adam(self.value_net.parameters(), lr=lr_actor, weight_decay=1e-4)
    
        self.MseLoss = nn.MSELoss()


    def update(self, memory):
        # Convert lists to tensors
        old_states = torch.stack(memory.states).detach()
        old_actions = torch.stack(memory.actions).detach()
        old_logprobs = torch.stack(memory.logprobs).detach()
        state_values = self.value_net(old_states)


        # Compute GAE advantages and returns
        advantages = []
        returns = []
        gae = 0
        values = list(state_values) + [torch.tensor(0.0).to(self.device)]
        for t in reversed(range(len(memory.rewards))):
            mask = 0 if memory.is_terminals[t] else 1
            delta = memory.rewards[t] + self.gamma * values[t+1] * mask - values[t]
            gae = delta + self.gamma * self.lam * mask * gae
            advantages.insert(0, gae)
            returns.insert(0, gae + values[t])

        advantages = torch.tensor(advantages, dtype=torch.float32).to(self.device)
        returns = torch.tensor(returns, dtype=torch.float32).to(self.device)
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        # Optimize for K epochs
        for _ in range(self.K_epochs):
            logprobs, entropy = self.policy.evaluate(old_states, old_actions)
            state_values = self.value_net(old_states)

            critic_loss = self.MseLoss(state_values, returns)
            self.value_optimizer.zero_grad()
            critic_loss.backward()
            self.value_optimizer.step()


            ratios = torch.exp(logprobs - old_logprobs)
            surr1 = ratios * advantages
            surr2 = torch.clamp(ratios, 1 - self.eps_clip, 1 + self.eps_clip) * advantages
            actor_loss = -torch.min(surr1, surr2).mean()
            entropy_loss = -entropy.mean()
            policy_loss = actor_loss + 0.01 * entropy_loss

            self.policy_optimizer.zero_grad()
            policy_loss.backward()
            self.policy_optimizer.step()

        # Update old policy
        self.policy_old.load_state_dict(self.policy.state_dict())
        memory.clear()

# Memory for Rollouts
class Memory:
    def __init__(self):
        self.states = []
        self.actions = []
        self.logprobs = []
        self.rewards = []
        self.is_terminals = []
        self.values = []

    def clear(self):
        del self.states[:], self.actions[:], self.logprobs[:]
        del self.rewards[:], self.is_terminals[:], self.values[:]

if __name__ == '__main__':
    # Hyperparameters
    env = gym.make('Pendulum-v1')
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.shape[0]
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    ppo = PPO(state_dim, action_dim, device=device)
    memory = Memory()

    max_episodes = 2000
    update_timestep = 2000
    log_interval = 20

    timestep, running_reward, avg_length = 0, 0, 0
    for i in range(1, max_episodes+1):
        state, _ = env.reset()
        done, trunc = False, False
        t = 0
        while not (done or trunc):
            timestep += 1

            state_tensor = torch.FloatTensor(state).to(ppo.device)
            action, logprob = ppo.policy_old.act(state_tensor)
            state, reward, done, trunc, info = env.step(action.cpu().numpy())
            
            memory.states.append(state_tensor)
            memory.actions.append(action)
            memory.logprobs.append(logprob)
            memory.rewards.append(reward)
            memory.is_terminals.append(done)

            running_reward += reward
            if done: break

            if timestep % update_timestep == 0:
                ppo.update(memory)
                timestep = 0
            t += 1
        avg_length += t
        if i % log_interval == 0:
            print(f'Episode {i} \t Avg length: {avg_length/log_interval:.2f} \t Avg reward: {running_reward/log_interval:.2f}')
            running_reward, avg_length = 0, 0

    torch.save(ppo.policy.state_dict(), 'ppo_pendulum.pth')
    env.close()

Episode 20 	 Avg length: 200.00 	 Avg reward: -1323.35
Episode 40 	 Avg length: 200.00 	 Avg reward: -1167.10
Episode 60 	 Avg length: 200.00 	 Avg reward: -1153.40
Episode 80 	 Avg length: 200.00 	 Avg reward: -1179.17
Episode 100 	 Avg length: 200.00 	 Avg reward: -1250.77
Episode 120 	 Avg length: 200.00 	 Avg reward: -1232.00
Episode 140 	 Avg length: 200.00 	 Avg reward: -1240.72
Episode 160 	 Avg length: 200.00 	 Avg reward: -1183.39
Episode 180 	 Avg length: 200.00 	 Avg reward: -1227.35
Episode 200 	 Avg length: 200.00 	 Avg reward: -1127.25
Episode 220 	 Avg length: 200.00 	 Avg reward: -1107.09
Episode 240 	 Avg length: 200.00 	 Avg reward: -933.86


KeyboardInterrupt: 

# modified

In [4]:
import gym
import numpy as np
import torch
import torch.nn as nn
from torch.distributions import Normal

from syn_rl.network.policy import GaussianPolicyNetwork
from syn_rl.network.value import ValueNetwork


# PPO Agent with Clipped Surrogate and GAE
class PPO:
    def __init__(self, state_dim, action_dim, lr=3e-4,
                 gamma=0.99, lam=0.95,
                 K_epochs=80, eps_clip=0.2,
                 device='cpu'):
        self.gamma = gamma
        self.lam = lam
        self.eps_clip = eps_clip
        self.K_epochs = K_epochs
        self.device = torch.device(device)

        self.policy = GaussianPolicyNetwork(state_dim, action_dim, [128]).to(device)
        self.policy_old = GaussianPolicyNetwork(state_dim, action_dim, [128]).to(device)
        self.policy_old.load_state_dict(self.policy.state_dict())
        self.value_net = ValueNetwork(state_dim, [128]).to(device)

        self.policy_optimizer = torch.optim.Adam(self.policy.parameters(), lr=lr, weight_decay=1e-4)
        self.value_optimizer = torch.optim.Adam(self.value_net.parameters(), lr=lr, weight_decay=1e-4)
    
        self.MseLoss = nn.MSELoss()


    def update(self, memory):
        # Convert lists to tensors
        old_states = torch.stack(memory.states).detach()
        old_actions = torch.stack(memory.actions).detach()
        old_logprobs = torch.stack(memory.logprobs).detach()
        state_values = self.value_net(old_states)


        # Compute GAE advantages and returns
        advantages = []
        returns = []
        gae = 0
        values = list(state_values) + [torch.tensor(0.0).to(self.device)]
        for t in reversed(range(len(memory.rewards))):
            mask = 0 if memory.is_terminals[t] else 1
            delta = memory.rewards[t] + self.gamma * values[t+1] * mask - values[t]
            gae = delta + self.gamma * self.lam * mask * gae
            advantages.insert(0, gae)
            returns.insert(0, gae + values[t])

        advantages = torch.tensor(advantages, dtype=torch.float32).to(self.device)
        returns = torch.tensor(returns, dtype=torch.float32).to(self.device)
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        # Optimize for K epochs
        for _ in range(self.K_epochs):
            logprobs, entropy = self.policy.evaluate(old_states, old_actions)
            state_values = self.value_net(old_states)

            critic_loss = self.MseLoss(state_values, returns)
            self.value_optimizer.zero_grad()
            critic_loss.backward()
            self.value_optimizer.step()


            ratios = torch.exp(logprobs - old_logprobs)
            surr1 = ratios * advantages
            surr2 = torch.clamp(ratios, 1 - self.eps_clip, 1 + self.eps_clip) * advantages
            actor_loss = -torch.min(surr1, surr2).mean()
            entropy_loss = -entropy.mean()
            policy_loss = actor_loss + 0.01 * entropy_loss

            self.policy_optimizer.zero_grad()
            policy_loss.backward()
            self.policy_optimizer.step()

        # Update old policy
        self.policy_old.load_state_dict(self.policy.state_dict())
        memory.clear()

# Memory for Rollouts
class Memory:
    def __init__(self):
        self.states = []
        self.actions = []
        self.logprobs = []
        self.rewards = []
        self.is_terminals = []
        self.values = []

    def clear(self):
        del self.states[:], self.actions[:], self.logprobs[:]
        del self.rewards[:], self.is_terminals[:], self.values[:]

if __name__ == '__main__':
    # Hyperparameters
    env = gym.make('Pendulum-v1')
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.shape[0]
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    ppo = PPO(state_dim, action_dim, device=device)
    memory = Memory()

    max_episodes = 2000
    update_timestep = 2000
    log_interval = 20

    timestep, running_reward, avg_length = 0, 0, 0
    for i in range(1, max_episodes+1):
        state, _ = env.reset()
        done, trunc = False, False
        t = 0
        while not (done or trunc):
            timestep += 1

            state_tensor = torch.FloatTensor(state).to(ppo.device)
            action, logprob = ppo.policy_old.select_action(state_tensor)
            state, reward, done, trunc, info = env.step(action.detach().cpu().numpy())
            
            memory.states.append(state_tensor)
            memory.actions.append(action)
            memory.logprobs.append(logprob)
            memory.rewards.append(reward)
            memory.is_terminals.append(done)

            running_reward += reward
            if done: break

            if timestep % update_timestep == 0:
                ppo.update(memory)
                timestep = 0
            t += 1
        avg_length += t
        if i % log_interval == 0:
            print(f'Episode {i} \t Avg length: {avg_length/log_interval:.2f} \t Avg reward: {running_reward/log_interval:.2f}')
            running_reward, avg_length = 0, 0

    torch.save(ppo.policy.state_dict(), 'ppo_pendulum.pth')
    env.close()

c:\Users\HeydarianArdakaniA\AppData\Local\anaconda3\Lib\site-packages\gym\utils\passive_env_checker.py:233: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):
c:\Users\HeydarianArdakaniA\AppData\Local\anaconda3\Lib\site-packages\torch\nn\modules\loss.py:538: UserWarning: Using a target size (torch.Size([2000])) that is different to the input size (torch.Size([2000, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


Episode 20 	 Avg length: 200.00 	 Avg reward: -1332.13
Episode 40 	 Avg length: 200.00 	 Avg reward: -1309.41
Episode 60 	 Avg length: 200.00 	 Avg reward: -1312.96
Episode 80 	 Avg length: 200.00 	 Avg reward: -1399.38
Episode 100 	 Avg length: 200.00 	 Avg reward: -1296.55
Episode 120 	 Avg length: 200.00 	 Avg reward: -1426.51
Episode 140 	 Avg length: 200.00 	 Avg reward: -1378.30
Episode 160 	 Avg length: 200.00 	 Avg reward: -1428.01
Episode 180 	 Avg length: 200.00 	 Avg reward: -1525.27
Episode 200 	 Avg length: 200.00 	 Avg reward: -1415.87
Episode 220 	 Avg length: 200.00 	 Avg reward: -1410.73
Episode 240 	 Avg length: 200.00 	 Avg reward: -1484.04
Episode 260 	 Avg length: 200.00 	 Avg reward: -1410.44
Episode 280 	 Avg length: 200.00 	 Avg reward: -1427.64
Episode 300 	 Avg length: 200.00 	 Avg reward: -1393.83
Episode 320 	 Avg length: 200.00 	 Avg reward: -1423.48
Episode 340 	 Avg length: 200.00 	 Avg reward: -1457.08
Episode 360 	 Avg length: 200.00 	 Avg reward: -1562

KeyboardInterrupt: 

In [6]:
from syn_rl.utils.buffer import RolloutBuffer
import numpy as np
from collections import deque
import random
import itertools

# Assuming RolloutBuffer class is defined as provided
# Create buffer
buffer_size = 100
batch_size = 5
rollout_buffer = RolloutBuffer(buffer_size)

# Sample experiences: (state, action, reward, next_state, done)
exp1 = (np.array([1, 2]), 1, 0.5, np.array([3, 4]), False)
exp2 = (np.array([3, 4]), 2, 1.0, np.array([5, 6]), True)
exp3 = (np.array([5, 6]), 3, 1.5, np.array([7, 8]), False)

# Test 1: Initialization
print("Test 1: Initialization")
print(f"Buffer length: {len(rollout_buffer)} (Expected: 0)")
print(f"Buffer type: {type(rollout_buffer.buffer)} (Expected: deque)")
print(f"Max length: {rollout_buffer.buffer.maxlen} (Expected: {buffer_size})")
print()

# Test 2: Push
print("Test 2: Push")
rollout_buffer.push(exp1)
print(f"Length after push: {len(rollout_buffer)} (Expected: 1)")
print(f"First experience: {rollout_buffer[0]} (Expected: {exp1})")

# Test buffer size limit
for i in range(buffer_size + 10):
    rollout_buffer.push(exp2)
print(f"Length after overfill: {len(rollout_buffer)} (Expected: {buffer_size})")
print()

# Test 3: Sample basic
print("Test 3: Sample basic")
rollout_buffer.clear()
for _ in range(10):
    rollout_buffer.push(exp1)
    rollout_buffer.push(exp2)
    rollout_buffer.push(exp3)
batch = rollout_buffer.sample(batch_size)
print(f"Batch components: {len(batch)} (Expected: 5)")
print(f"State batch shape: {batch[0].shape} (Expected: ({batch_size}, 2))")
print(f"Action batch shape: {batch[1].shape} (Expected: ({batch_size}, 1))")
print(f"All components are numpy arrays: {all(isinstance(x, np.ndarray) for x in batch)}")
print()

# Test 4: Sample with next state
print("Test 4: Sample with next state")
batch = rollout_buffer.sample(batch_size, include_next_state=True)
print(f"State batch length: {len(batch[0])} (Expected: {batch_size + 1})")
print(f"Other components length: {len(batch[1])} (Expected: {batch_size})")
print()

# Test 5: Sample all
print("Test 5: Sample all")
rollout_buffer.clear()
rollout_buffer.push(exp1)
rollout_buffer.push(exp2)
rollout_buffer.push(exp3)
batch = rollout_buffer.sample(batch_size, return_all=True)
print(f"Batch components: {len(batch)} (Expected: 5)")
print(f"State batch length: {len(batch[0])} (Expected: 3)")
print()

# Test 6: Clear
print("Test 6: Clear")
rollout_buffer.clear()
print(f"Length after clear: {len(rollout_buffer)} (Expected: 0)")
print()

# Test 7: Error cases
print("Test 7: Error cases")
try:
    rollout_buffer.sample(batch_size)
    print("Error: Should have failed on empty buffer")
except ValueError:
    print("Passed: Empty buffer error caught")
rollout_buffer.push(exp1)
try:
    rollout_buffer.sample(buffer_size + 1)
    print("Error: Should have failed on large batch size")
except ValueError:
    print("Passed: Large batch size error caught")

Test 1: Initialization
Buffer length: 0 (Expected: 0)
Buffer type: <class 'collections.deque'> (Expected: deque)
Max length: 100 (Expected: 100)

Test 2: Push
Length after push: 1 (Expected: 1)
First experience: (array([1, 2]), 1, 0.5, array([3, 4]), False) (Expected: (array([1, 2]), 1, 0.5, array([3, 4]), False))
Length after overfill: 100 (Expected: 100)

Test 3: Sample basic
Batch components: 5 (Expected: 5)
State batch shape: (5, 2) (Expected: (5, 2))
Action batch shape: (5, 1) (Expected: (5, 1))
All components are numpy arrays: True

Test 4: Sample with next state
State batch length: 6 (Expected: 6)
Other components length: 5 (Expected: 5)

Test 5: Sample all
Batch components: 5 (Expected: 5)
State batch length: 3 (Expected: 3)

Test 6: Clear
Length after clear: 0 (Expected: 0)

Test 7: Error cases
Passed: Empty buffer error caught
Passed: Large batch size error caught


In [10]:
# Test 3: Sample basic
print("Test 3: Sample basic")
rollout_buffer.clear()
for _ in range(10):
    rollout_buffer.push(exp1)
    rollout_buffer.push(exp2)
    rollout_buffer.push(exp3)
batch = rollout_buffer.sample(batch_size)
print(f"Batch components: {len(batch)} (Expected: 5)")
print(f"State batch shape: {batch[0].shape} (Expected: ({batch_size}, 2))")
print(f"Action batch shape: {batch[1].shape} (Expected: ({batch_size}, 1))")
print(f"All components are numpy arrays: {all(isinstance(x, np.ndarray) for x in batch)}")
print()


Test 3: Sample basic
Batch components: 5 (Expected: 5)
State batch shape: (5, 2) (Expected: (5, 2))
Action batch shape: (5, 1) (Expected: (5, 1))
All components are numpy arrays: True



In [13]:
rollout_buffer.sample(5, include_next_state=True)

[array([[1, 2],
        [3, 4],
        [5, 6],
        [1, 2],
        [3, 4],
        [5, 6]]),
 array([[1],
        [2],
        [3],
        [1],
        [2]]),
 array([[0.5],
        [1. ],
        [1.5],
        [0.5],
        [1. ]]),
 array([[3, 4],
        [5, 6],
        [7, 8],
        [3, 4],
        [5, 6]]),
 array([[False],
        [ True],
        [False],
        [False],
        [ True]])]